In [1]:
import torch
print("device_count:", torch.cuda.device_count())
print("is_available:", torch.cuda.is_available())

device_count: 1
is_available: True


In [2]:
import sys, site
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
from datasets import Dataset
import json

In [3]:
with open("train.json", "r") as f:
    train_dataset = Dataset.from_list(json.load(f))
with open("validation.json", "r") as f:
    validation_dataset = Dataset.from_list(json.load(f))
print(f"Train: {len(train_dataset)} examples, Validation: {len(validation_dataset)} examples")

Train: 460 examples, Validation: 101 examples


In [4]:
def basic_formatting_function(row):
    import json
    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    with open(f"./schemas/{clean_id}.json") as f:
        schema_data = json.load(f)
    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        schema[schema_data["table_names_original"][i]].append(name)
    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema: {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return a JSON object with only the relevant tables as keys and lists of relevant column names as values. "
        "You MUST include specific column names — do not return empty lists unless a table has no relevant columns. "
        "Example: {\"Orders\": [\"order_id\", \"total\"], \"Customers\": [\"name\"]}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {"text": f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"}


def pkfk_formatting_function(row):
    import json
    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    with open(f"./schemas/{clean_id}.json") as f:
        schema_data = json.load(f)
    column_info = schema_data["column_names_original"]
    primary_keys = schema_data.get("primary_keys", [])
    foreign_keys = schema_data.get("foreign_keys", [])
    col_annotations = {}
    for pk in primary_keys:
        if isinstance(pk, list):
            for col_idx in pk:
                col_annotations[col_idx] = "(PK)"
        else:
            col_annotations[pk] = "(PK)"
    for from_idx, to_idx in foreign_keys:
        to_table_name = schema_data["table_names_original"][column_info[to_idx][0]]
        if from_idx in col_annotations and col_annotations[from_idx] == "(PK)":
            col_annotations[from_idx] = f"(PK,FK\u2192{to_table_name})"
        else:
            col_annotations[from_idx] = f"(FK\u2192{to_table_name})"
    schema = {}
    for col_idx, (table_idx, col_name) in enumerate(column_info):
        if table_idx == -1:
            continue
        table_name = schema_data["table_names_original"][table_idx]
        if table_name not in schema:
            schema[table_name] = []
        ann = col_annotations.get(col_idx, "")
        schema[table_name].append(f"{col_name} {ann}" if ann else col_name)
    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema with PK/FK annotations, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists (without annotations in the output)."
    )
    prompt = (
        f"Database schema (PK/FK annotated): {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only — column names without annotations: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {"text": f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"}


def sorted_formatting_function(row):
    import json
    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    with open(f"./schemas/{clean_id}.json") as f:
        schema_data = json.load(f)
    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        schema[schema_data["table_names_original"][i]].append(name)
    sorted_schema = {t: sorted(cols) for t, cols in sorted(schema.items())}
    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    # used coding agent to improve prompt 
    prompt = (
        f"Database schema (sorted): {sorted_schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only in this format: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {"text": f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"}

In [5]:
experiment = Experiment(experiment_name="augmented_final_experiments_rf", mode="fit")

The previously running experiment augmented_final_experiments_s was forcibly ended. Created a new experiment 'augmented_final_experiments_rf' with Experiment ID: 51 and Metric Experiment ID: augmented_final_experiments_rf at /home/sjrao/rapidfireai/rapidfire_experiments/augmented_final_experiments_rf


In [6]:
import torch

QWEN = "Qwen/Qwen2.5-1.5B-Instruct"
ATTN = ["q_proj", "k_proj", "v_proj", "o_proj"]
ALL = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

configs_spec = [
    ("P1_r4_attn_lr1e5_e2", basic_formatting_function, 4, 8, 1e-5, ATTN, "linear", 2),
    ("P2_r4_attn_lr1e5_e3", basic_formatting_function, 4, 8, 1e-5, ATTN, "linear", 3),
    ("P6_r4_attn_lr5e6_e2", basic_formatting_function, 4, 8, 5e-6, ATTN, "linear", 2),
    ("Q1_r4_attn_lr2e5_e2", basic_formatting_function, 4, 8, 2e-5, ATTN, "linear", 2),
    ("Q5_r4_attn_lr1e5_cosine_e3", basic_formatting_function, 4, 8, 1e-5, ATTN, "cosine", 3),
    ("R9_pkfk_r8_attn_lr2e5_e3", pkfk_formatting_function, 4, 8, 1e-5, ATTN, "linear", 2),
    ("R10_sorted_r8_attn_lr2e5_e3", sorted_formatting_function, 4, 8, 1e-5, ATTN, "linear", 2),
]

all_configs = []
for label, fmt_func, r, alpha, lr, target_modules, scheduler, epochs in configs_spec:
    all_configs.append(RFModelConfig(
        model_name=QWEN,
        peft_config=RFLoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.1, target_modules=target_modules, bias="none"),
        training_args=RFSFTConfig(
            learning_rate=lr,
            lr_scheduler_type=scheduler,
            num_train_epochs=epochs,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            gradient_checkpointing=False,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=20,
            bf16=False,
            fp16=True,
        ),
        model_type="causal_lm",
        model_kwargs={"torch_dtype": torch.float16, "use_cache": False},
        formatting_func=fmt_func,
    ))

print(f"Total configs: {len(all_configs)}")
for i, (label, *_) in enumerate(configs_spec):
    print(f"  [{i+1}] {label}")
config_set = List(all_configs)

Total configs: 7
  [1] P1_r4_attn_lr1e5_e2
  [2] P2_r4_attn_lr1e5_e3
  [3] P6_r4_attn_lr5e6_e2
  [4] Q1_r4_attn_lr2e5_e2
  [5] Q5_r4_attn_lr1e5_cosine_e3
  [6] R9_pkfk_r8_attn_lr2e5_e3
  [7] R10_sorted_r8_attn_lr2e5_e3


In [7]:
def sample_create_model(model_config):
    import gc, os
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model_kwargs = dict(model_config["model_kwargs"])
    model_kwargs.pop("device_map", None)
    model_kwargs.setdefault("low_cpu_mem_usage", True)
    model = AutoModelForCausalLM.from_pretrained(model_config["model_name"], **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_config["model_name"])
    return (model, tokenizer)

In [8]:
config_group = RFGridSearch(configs=config_set, trainer_type="SFT")

In [9]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    validation_dataset,
    num_chunks=1,
    seed=42,
)

/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/__init__.py:22: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_client.py:40: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


INFO 06-03 19:53:29 [__init__.py:216] Automatically detected platform cuda.


/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_generation.py:41: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


Started 1 worker processes successfully
Created workers


/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/__init__.py:22: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_client.py:40: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_generation.py:41: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


In [11]:
experiment.end()

No experiment is currently running. Nothing to end.
Workers stopped
